# Forge-Neo Colab

Runs Stable Diffusion WebUI Forge - Neo on Colab, imports models from Civitai / Hugging Face / Google Drive, and exposes the UI through Gradio or ngrok.

In [ ]:
# @title Settings
Helper_URL = "https://raw.githubusercontent.com/YOUR_REPO/YOUR_BRANCH/forge_neo_colab/forge_neo_colab.py" # @param {type:"string"}
Forge_Ref = "neo" # @param {type:"string", placeholder:"tag, commit SHA, or branch name"}
Forge_Neo_Ref = "latest-tag" # @param {type:"string"}
Civitai_Token = "" # @param {type:"string"}
HF_Token = "" # @param {type:"string"}
NGROK_Token = "" # @param {type:"string"}
Gradio_Auth = "" # @param {type:"string", placeholder:"user:strong_password"}
Tunnel = "gradio" # @param ["gradio", "ngrok", "cloudflared", "none"]
Mount_Google_Drive = "No" # @param ["No", "Yes"]

import os

if Forge_Ref:
    os.environ["FORGE_NEO_REF"] = Forge_Ref
if Forge_Neo_Ref:
    os.environ["FORGE_NEO_REF"] = Forge_Neo_Ref
if Civitai_Token:
    os.environ["CIVITAI_TOKEN"] = Civitai_Token
if HF_Token:
    os.environ["HF_TOKEN"] = HF_Token
if NGROK_Token:
    os.environ["NGROK_TOKEN"] = NGROK_Token


In [ ]:
# @title Load Helper
!curl -L -o /content/forge_neo_colab.py "$Helper_URL"
%run /content/forge_neo_colab.py paths


In [ ]:
# @title Install Forge-Neo
%run /content/forge_neo_colab.py install --ref "$Forge_Ref"


In [ ]:
# @title Optional Google Drive Folders
if Mount_Google_Drive == "Yes":
    from google.colab import drive
    from pathlib import Path
    drive.mount("/content/drive")

    base = Path("/content/drive/MyDrive/ForgeNeo")
    links = {
        base / "checkpoints": CKPT / "drive_checkpoints",
        base / "loras": LORA / "drive_loras",
        base / "vae": VAE / "drive_vae",
    }
    for source, target in links.items():
        source.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            target.symlink_to(source, target_is_directory=True)
        print(f"{target} -> {source}")
else:
    print("Google Drive mount skipped.")


## Model Imports

Use direct cells below, or create `/content/models.json` and run the manifest cell.

In [ ]:
# @title Direct Model Downloads
# Examples. Replace or add your own lines.
# %fn_download ckpt https://huggingface.co/user/repo/resolve/main/model.safetensors
# %fn_download lora https://civitai.com/models/122359/detail-tweaker-xl
# %fn_download vae https://drive.google.com/file/d/FILE_ID/view?usp=sharing


In [ ]:
# @title Download From /content/models.json
from pathlib import Path

manifest = Path("/content/models.json")
if manifest.exists():
    %run /content/forge_neo_colab.py download --manifest /content/models.json
else:
    print("/content/models.json not found. Skip manifest download.")


In [ ]:
# @title Launch Forge-Neo
auth_args = ["--gradio-auth", Gradio_Auth] if Gradio_Auth else []
cmd = ["/content/forge_neo_colab.py", "launch", "--tunnel", Tunnel, "--", *auth_args]

import runpy, sys
old_argv = sys.argv[:]
try:
    sys.argv = cmd
    runpy.run_path("/content/forge_neo_colab.py", run_name="__main__")
finally:
    sys.argv = old_argv
